# Cycle Analysis — 260121 CZA-flueCO2 Experiment

Validate the cycle detection and integration pipeline on the 260121 dataset.

In [ ]:
import sys
from pathlib import Path

# Add backend to path so we can import effi
sys.path.insert(0, str(Path.cwd().parent / "backend"))

import pandas as pd
import plotly.graph_objects as go
from effi import (
    load_experiment,
    detect_cycles,
    analyze_experiment,
)
from effi.integration import NATIVE_SPECIES

## 1. Load Data

In [ ]:
data_dir = Path.cwd().parent / "260121_CZA-flueCO2"

reactor_files = sorted(data_dir.glob("ExportData*.txt"))
reactor_files = [str(f) for f in reactor_files]
print(f"Reactor files: {len(reactor_files)}")
for f in reactor_files:
    print(f"  {Path(f).name}")

ir_file = str(data_dir / "260121_Data_All.csv")
oxygen_file = str(data_dir / "260121_oxygen.csv")

df = load_experiment(
    reactor_files,
    ir_file,
    oxygen_file,
    offset=pd.Timedelta("5h"),
)
print(f"\nMerged DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Time range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")

## 2. Detect Cycles

In [ ]:
cycles = detect_cycles(df)
print(f"Detected {len(cycles)} cycles\n")

# Print summary table
cycle_summary = []
for c in cycles:
    cycle_summary.append({
        "cycle": c.cycle_id,
        "hp_start": c.high_p.start,
        "hp_end": c.high_p.end,
        "lp_start": c.low_p.start,
        "lp_end": c.low_p.end,
        "hp_rows": c.high_p.end_idx - c.high_p.start_idx + 1,
        "lp_rows": c.low_p.end_idx - c.low_p.start_idx + 1,
    })
pd.DataFrame(cycle_summary)

## 3. Integrate Species

In [ ]:
results = analyze_experiment(df, cycles)
print(f"Results: {results.shape[0]} rows\n")

# Check for negative areas or NaN
n_negative = (results[["high_p_area", "low_p_area"]] < 0).sum().sum()
n_nan = results[["high_p_area", "low_p_area"]].isna().sum().sum()
print(f"Negative areas: {n_negative}")
print(f"NaN areas: {n_nan}\n")

# Show results for key product species
key_species = ["Methanol", "Ethanol", "Dimethyl Ether", "Carbon Dioxide", "Water"]
results[results["species"].isin(key_species)]

## 4. Visualize Cycles

Plot a single cycle with fill-between shading for high-P and low-P windows.

In [ ]:
def plot_cycle(df, cycle, species=None):
    """Plot one cycle with window shading."""
    if species is None:
        species = ["Methanol (%)", "Dimethyl Ether (%)", "Carbon Dioxide (%)"]

    # Pad view by 2 minutes on each side
    pad = pd.Timedelta("2min")
    t_start = cycle.high_p.start - pad
    t_end = cycle.low_p.end + pad
    mask = (df["Timestamp"] >= t_start) & (df["Timestamp"] <= t_end)
    view = df.loc[mask]

    fig = go.Figure()

    # High-P shading
    fig.add_vrect(
        x0=cycle.high_p.start, x1=cycle.high_p.end,
        fillcolor="rgba(0,100,255,0.1)", line_width=0,
        annotation_text="High P", annotation_position="top left",
    )
    # Low-P shading
    fig.add_vrect(
        x0=cycle.low_p.start, x1=cycle.low_p.end,
        fillcolor="rgba(255,100,0,0.1)", line_width=0,
        annotation_text="Low P", annotation_position="top left",
    )

    for col in species:
        if col in view.columns:
            fig.add_trace(go.Scatter(
                x=view["Timestamp"], y=view[col],
                mode="lines", name=col,
            ))

    # Add reactor conditions on secondary y-axis
    for col, dash in [("Reactor P RSP", "dash"), ("Reactor T RSP", "dot")]:
        if col in view.columns:
            fig.add_trace(go.Scatter(
                x=view["Timestamp"], y=view[col],
                mode="lines", name=col,
                line=dict(dash=dash),
                yaxis="y2",
            ))

    fig.update_layout(
        title=f"Cycle {cycle.cycle_id}",
        xaxis_title="Time",
        yaxis=dict(title="Concentration (%)"),
        yaxis2=dict(title="RSP (°C / bar)", overlaying="y", side="right"),
        hovermode="x unified",
        height=500,
    )
    return fig

In [ ]:
# Verify 3 cycles: early, middle, late
for idx in [0, len(cycles) // 2, len(cycles) - 1]:
    c = cycles[idx]
    print(f"\nCycle {c.cycle_id}: HP {c.high_p.start} → {c.high_p.end}, "
          f"LP {c.low_p.start} → {c.low_p.end}")
    fig = plot_cycle(df, c)
    fig.show()

## 5. Sanity Checks

In [ ]:
# Check STATUS columns are unaffected
status_cols = [c for c in df.columns if "STATUS" in c]
for col in status_cols:
    unique_vals = df[col].dropna().unique()
    print(f"{col}: unique values = {sorted(unique_vals)}")

print()

# Check that integration results are positive and reasonable
pivot = results.pivot_table(
    index="cycle_id",
    columns="species",
    values="high_p_area",
)
print("High-P area summary (all cycles):")
pivot[["Methanol", "Ethanol", "Dimethyl Ether"]].describe()